# PROMPT 4: Dense Retrieval

**Amaç:** 47k chunks'ı embed et ve FAISS index'le

## Pipeline
```
Chunks (47k)  ← Prompt 3 çıktısı
     ↓
Embedding model (sentence-transformers)
     ↓
Dense vectors (47k × 384-dim)
     ↓
FAISS index (IndexFlatL2)
     ↓
Fast similarity search (<50ms)
     ↓
Top-5 chunks  → Prompt 5 (Reranking) için
```

## Temel Kavramlar

### Dense Vector
> Text'i sayılara dönüştür
> - "Kasten adam öldürme" → [0.145, -0.234, 0.567, ...] (384 sayı)
> - Semantically similar texts = similar vectors

### Embedding Model
> Text → Vector dönüştürücü
> - distiluse-base-multilingual-cased-v2
> - Turkish support + fast (~50ms per chunk)
> - 384-dimensional output

### Cosine Similarity
> Vektörler arasındaki benzerlik (0-1, 1=identical)
> - Query vector vs Chunk vector
> - Find top-k most similar chunks

In [ ]:
import sys
import json
import time
import logging
import numpy as np
from pathlib import Path
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Config
DATA_DIR = Path.cwd().parent / 'data'
CHUNKED_DIR = DATA_DIR / 'chunked'
MODEL_DIR = Path.cwd().parent / 'models' / 'retrieval_index'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Paths:")
print(f"   Chunked dir: {CHUNKED_DIR}")
print(f"   Model dir: {MODEL_DIR}")

## Cell 2: Import DenseRetriever

In [ ]:
from retrieval.dense import DenseRetriever, setup_dense_retriever

print(f"✅ DenseRetriever imported")

## Cell 3: Load Chunks from Prompt 3

In [ ]:
# Find chunked JSONL files
chunked_files = list(CHUNKED_DIR.glob('*_chunked.jsonl'))

print(f"📂 Chunked files found: {len(chunked_files)}")

# Load all chunks
all_chunks = []

for file_path in chunked_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            chunk = json.loads(line.strip())
            all_chunks.append(chunk)

print(f"✅ Total chunks loaded: {len(all_chunks):,}")

# Sample
sample_chunk = all_chunks[0]
print(f"\n📋 Sample chunk:")
print(f"  ID: {sample_chunk['chunk_id']}")
print(f"  Tokens: {sample_chunk['chunk_length_tokens']}")
print(f"  Text: {sample_chunk['chunk_text'][:100]}...")

## Cell 4: Embedding Model Demo

Show how embedding model works on sample texts.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load model (will cache locally)
print("Loading embedding model...")
model = SentenceTransformer('distiluse-base-multilingual-cased-v2')

# Get model info
embedding_dim = model.get_sentence_embedding_dimension()
print(f"\n✅ Model loaded:")
print(f"   Model: distiluse-base-multilingual-cased-v2")
print(f"   Embedding dimension: {embedding_dim}")

# Example embeddings
sample_texts = [
    "Kasten adam öldürme suçu nedir?",
    "Intention to kill in Turkish law",
    "Taksir halinde ölüm",
    "Madde 81 ne demek?"
]

embeddings = model.encode(sample_texts, normalize_embeddings=True)

print(f"\n📊 Sample embeddings:")
for i, text in enumerate(sample_texts):
    emb = embeddings[i]
    print(f"\nText {i+1}: {text[:40]}...")
    print(f"  Vector: [{emb[0]:.4f}, {emb[1]:.4f}, {emb[2]:.4f}, ..., {emb[-1]:.4f}]")
    print(f"  Norm: {np.linalg.norm(emb):.4f}  (normalized = 1.0)")

# Compute similarity
print(f"\n📈 Cosine Similarity:")
from scipy.spatial.distance import cosine

for i in range(len(sample_texts)):
    for j in range(i+1, len(sample_texts)):
        sim = 1 - cosine(embeddings[i], embeddings[j])
        print(f"  Text {i+1} vs Text {j+1}: {sim:.4f}")

## Cell 5: FAISS Index Creation

Embed all chunks and create FAISS index.

In [ ]:
# Initialize DenseRetriever
print("Initializing DenseRetriever...")

retriever = DenseRetriever(
    model_name='distiluse-base-multilingual-cased-v2',
    index_dir=str(MODEL_DIR),
    device='cpu'  # Use 'cuda' if GPU available
)

print(f"\n✅ DenseRetriever initialized")

## Cell 6: Embed All Chunks

This will take 1-2 minutes on CPU, 10-30s on GPU.

In [ ]:
# Start timing
start_time = time.time()

# Embed all chunks
print(f"Embedding {len(all_chunks):,} chunks...")
embeddings = retriever.embed_chunks(
    all_chunks,
    batch_size=32,
    show_progress=True
)

embedding_time = time.time() - start_time

print(f"\n✅ Embeddings complete:")
print(f"   Shape: {embeddings.shape}")
print(f"   Data type: {embeddings.dtype}")
print(f"   Memory: {embeddings.nbytes / (1024**2):.2f} MB")
print(f"   Time: {embedding_time:.2f}s")
print(f"   Speed: {len(all_chunks)/embedding_time:.0f} chunks/sec")

## Cell 7: Create FAISS Index

In [ ]:
# Create FAISS index
print("Creating FAISS index...")

start_time = time.time()
index = retriever.create_index(
    all_chunks,
    embeddings,
    index_type='flat'  # Brute-force, but accurate
)
index_time = time.time() - start_time

print(f"\n✅ Index created:")
print(f"   Index type: IndexFlatL2")
print(f"   Total vectors: {index.ntotal:,}")
print(f"   Dimension: {embeddings.shape[1]}")
print(f"   Time: {index_time:.2f}s")

# Check saved files
print(f"\n📁 Index files:")
for f in sorted(MODEL_DIR.glob('*')):
    size_mb = f.stat().st_size / (1024**2) if f.is_file() else 0
    if f.is_file():
        print(f"   {f.name}: {size_mb:.2f} MB")

## Cell 8: Test Search

In [ ]:
# Test queries
test_queries = [
    "Kasten adam öldürme suçu nedir?",
    "Taksir halinde ölüm cezası",
    "Turkish criminal law article 81",
    "Ağırlaştırıcı sebepler neler",
    "Hapis cezasının süresi"
]

print(f"Testing search with {len(test_queries)} queries...\n")

all_search_times = []

for query in test_queries:
    print(f"Query: {query}")
    
    # Search
    start = time.time()
    results = retriever.search(query, k=5)
    search_time = time.time() - start
    all_search_times.append(search_time)
    
    print(f"  Time: {search_time*1000:.2f}ms")
    print(f"  Results:")
    
    for result in results:
        print(f"    {result.rank}. [{result.score:.4f}] {result.chunk_text[:80]}...")
    
    print()

# Performance stats
print(f"\n⏱️  Search Performance:")
print(f"   Avg time: {np.mean(all_search_times)*1000:.2f}ms")
print(f"   Min time: {np.min(all_search_times)*1000:.2f}ms")
print(f"   Max time: {np.max(all_search_times)*1000:.2f}ms")

## Cell 9: Batch Search Example

Show how to search multiple queries efficiently.

In [ ]:
# Batch search
batch_queries = test_queries[:3]

print(f"Batch search for {len(batch_queries)} queries...\n")

start = time.time()
batch_results = retriever.batch_search(batch_queries, k=3)
batch_time = time.time() - start

print(f"Total time: {batch_time:.2f}s")
print(f"Per-query time: {batch_time/len(batch_queries)*1000:.2f}ms\n")

for query, results in zip(batch_queries, batch_results):
    print(f"Query: {query}")
    print(f"  Results: {len(results)} chunks")
    print()

# Export batch results to DataFrame
results_data = []
for query, results in zip(batch_queries, batch_results):
    for result in results:
        results_data.append({
            'query': query,
            'rank': result.rank,
            'score': result.score,
            'chunk_id': result.chunk_id,
            'law_name': result.metadata.get('law_name'),
            'article_no': result.metadata.get('article_no'),
        })

results_df = pd.DataFrame(results_data)
print(f"Results DataFrame:")
print(results_df)

## Cell 10: Generate Report

Save retrieval statistics and benchmark results.

In [ ]:
# Create comprehensive report
report = {
    'timestamp': datetime.now().isoformat(),
    'input': {
        'total_chunks': len(all_chunks),
        'source_file': str(CHUNKED_DIR / 'turkish_law_chunked.jsonl')
    },
    'embedding': {
        'model': 'distiluse-base-multilingual-cased-v2',
        'dimension': embeddings.shape[1],
        'dtype': str(embeddings.dtype),
        'embedding_time_seconds': embedding_time,
        'chunks_per_second': len(all_chunks) / embedding_time,
        'memory_mb': embeddings.nbytes / (1024**2)
    },
    'index': {
        'type': 'IndexFlatL2',
        'total_vectors': index.ntotal,
        'index_creation_time_seconds': index_time,
        'index_file_size_mb': (MODEL_DIR / 'dense.index').stat().st_size / (1024**2) if (MODEL_DIR / 'dense.index').exists() else 0
    },
    'search_benchmark': {
        'test_queries': len(test_queries),
        'results_per_query': 5,
        'avg_search_time_ms': np.mean(all_search_times) * 1000,
        'min_search_time_ms': np.min(all_search_times) * 1000,
        'max_search_time_ms': np.max(all_search_times) * 1000,
        'qps': 1 / np.mean(all_search_times)  # Queries per second
    },
    'total_processing_time_seconds': embedding_time + index_time
}

# Save report
report_file = MODEL_DIR / 'retrieval_report.json'
with open(report_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"✅ Report saved: {report_file}")

# Print report
print(f"\n" + "="*60)
print("DENSE RETRIEVAL REPORT")
print("="*60)
print(f"\n📥 INPUT:")
print(f"   Chunks: {report['input']['total_chunks']:,}")

print(f"\n🔢 EMBEDDING:")
print(f"   Model: {report['embedding']['model']}")
print(f"   Dimension: {report['embedding']['dimension']}")
print(f"   Time: {report['embedding']['embedding_time_seconds']:.2f}s")
print(f"   Speed: {report['embedding']['chunks_per_second']:.0f} chunks/sec")
print(f"   Memory: {report['embedding']['memory_mb']:.2f} MB")

print(f"\n📀 INDEX:")
print(f"   Type: {report['index']['type']}")
print(f"   Vectors: {report['index']['total_vectors']:,}")
print(f"   Size: {report['index']['index_file_size_mb']:.2f} MB")
print(f"   Creation time: {report['index']['index_creation_time_seconds']:.2f}s")

print(f"\n🔍 SEARCH BENCHMARK:")
print(f"   Avg latency: {report['search_benchmark']['avg_search_time_ms']:.2f}ms")
print(f"   Min latency: {report['search_benchmark']['min_search_time_ms']:.2f}ms")
print(f"   Max latency: {report['search_benchmark']['max_search_time_ms']:.2f}ms")
print(f"   QPS: {report['search_benchmark']['qps']:.2f} queries/sec")

print(f"\n⏱️  TOTAL TIME: {report['total_processing_time_seconds']:.2f}s")
print(f"\n" + "="*60)